## **PART 5. DATA MODELLING**

In [1]:
#import library
import pandas as pd
import numpy as np
from sklearn import preprocessing

In [2]:
# Load the processed dataset
data = pd.read_csv("D:\\TAI LIEU DOWNLOAD\\data_processed.csv")

In [3]:
data.head()

,Product Name,Category,Dosage Form,Price,Trademark,Brand Origin,Country,Rating,Continent,General_function
0,"Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...",Chăm sóc cơ thể,Gel,105000.0,DECUMAR,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
1,Dung dịch vệ sinh vùng kín Bimunica 250ml dành...,Chăm sóc cơ thể,Dạng kem,230000.0,Eucerin,Hoa Kỳ,Liên Bang Nga,5.0,Europe,Chăm sóc cơ thể
2,"Kem giảm thâm vùng nách, mông, bikini Neothera...",Chăm sóc cơ thể,Dạng kem,139000.0,La Beauty,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
3,Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...,Chăm sóc cơ thể,Dạng kem,390000.0,SVR,Pháp,Pháp,unknown,Europe,Chăm sóc cơ thể
4,Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...,"Lăn khử mùi, xịt khử mùi",Dạng bọt,96000.0,Eucerin,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể


In [4]:
data.shape

(1999, 10)

Use LabelEncoder to convert categorical columns into numeric format because most machine learning models cannot process string data directly.

In [5]:
# Create a label encoder object for categorical features
label_encoder = preprocessing.LabelEncoder()

data['Country'] = label_encoder.fit_transform(data['Country'])
data['Trademark'] = label_encoder.fit_transform(data['Trademark'])
data['General_function'] = label_encoder.fit_transform(data['General_function'])
data['Dosage Form'] = label_encoder.fit_transform(data['Dosage Form'])
print(data.head())

                                        Product Name  \
0  Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1  Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2  Kem giảm thâm vùng nách, mông, bikini Neothera...   
3  Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4  Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   

                   Category  Dosage Form     Price  Trademark Brand Origin  \
0           Chăm sóc cơ thể           16  105000.0         75     Việt Nam   
1           Chăm sóc cơ thể           12  230000.0        118       Hoa Kỳ   
2           Chăm sóc cơ thể           12  139000.0        217     Việt Nam   
3           Chăm sóc cơ thể           12  390000.0        375         Pháp   
4  Lăn khử mùi, xịt khử mùi            9   96000.0        118     Việt Nam   

   Country   Rating Continent  General_function  
0       38      5.0      Asia                 0  
1       16      5.0    Europe                 0  
2       38      5.0      Asia               

In [6]:
# Split the data into training and prediction datasets
train_data = data[data['Rating'] != 'unknown']  # Product with rating
prediction_data = data[data['Rating'] == 'unknown']  # Product with rating = "unknown"

In [7]:
#Check shape of data
train_data.shape, prediction_data.shape

((1220, 10), (779, 10))

### 2. Feature Selection

In [8]:
#These will be the independent variables
features = ['Price', 'Trademark', 'Country', 'General_function', 'Dosage Form']

### **Splitting dataset into X and y**

In [9]:
# Data for training and test
X = train_data[features]
y = train_data["Rating"]
# Data for prediction
X_test = prediction_data[features]
y_test = prediction_data["Rating"]

In [10]:
#Check
X.head()

,Price,Trademark,Country,General_function,Dosage Form
0,105000.0,75,38,0,16
1,230000.0,118,16,0,12
2,139000.0,217,38,0,12
4,96000.0,118,38,0,9
5,132000.0,118,11,1,12


In [11]:
#Check
y.head()

0    5.0
1    5.0
2    5.0
4    5.0
5    5.0
Name: Rating, dtype: object

##### **X,y -> X_train, y_train, X_valid, y_valid**

In [12]:
from sklearn.model_selection import train_test_split
# Split train and test model in 80/20
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state=2024)

In [13]:
X.shape, X_train.shape, X_valid.shape

((1220, 5), (976, 5), (244, 5))

### **Model training**

To choose an algorithm suitable for the data, we refer to instructions from the official Scikit-learn documentation, including the following criteria:
1. Type of problem:
    * This is a problem of predicting continuous value (the amount of customer reviews), so it belongs to the group of regression problems.
2. Data scale:
    * With the current data size less than 100K samples, we choose algorithms suitable for small or medium data.
    
-> Considered regression algorithm:

- Random Forest Regression: Powerful in handling complex data and does not require data normalization.
- Gradient Boosting (XGBoost): Effective for problems that require accurate prediction, capable of handling outliers well.
- Ridge Regression: Suitable for small data, add regularization to avoid overfitting.
- SVR(kernel=rbf): Good for small problems that require accurate predictions, but can be slow when the data is large.
- ElasticNet Regression: Combines L1 and L2 regularization, handles well with data with many unrelated features.

In [14]:
#import library
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,  make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

### **Optimize hyperparameters and compare the evaluation of different models.**

#### **Objective**
Optimize hyperparameters for each model and evaluate their performance based on important metrics such as MSE, RMSE, training time, and memory usage.

#### **1. Definition and Importance of Hyperparameter Tuning**

Hyperparameter tuning is the process of searching for the optimal set of hyperparameters for a machine learning algorithm. This is important because:
- It helps the model achieve the best performance on the dataset.
- It prevents overfitting or underfitting.
- It ensures the model is well-tuned for the specific problem at hand.

#### **2. Steps to Follow**

##### **a. Prepare a list of models for tuning.**

In [15]:
# Normalize the feature data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Regressor models to be tested
models = [
    RandomForestRegressor(random_state=2024),
    GradientBoostingRegressor(random_state=2024),
    SVR(),
    Ridge(),
    ElasticNet()
]

##### **b. Define the hyperparameter search space.**

In [16]:
# Hyperparameter grids for each model
param_grids = {
    'RandomForestRegressor': {
        'n_estimators': [100, 200, 500],  # Number of trees in the forest
        'max_depth': [10, 20, 50, None],   # Maximum depth of the trees
        'min_samples_split': [2, 5, 10],   # Minimum number of samples required to split a node
        'min_samples_leaf': [1, 2, 4],     # Minimum number of samples in a leaf node
        'bootstrap': [True, False],        # Whether to use bootstrap samples or not
    },
    'GradientBoostingRegressor': {
        'n_estimators': [100, 200, 500],  # Number of trees
        'learning_rate': [0.1, 0.05, 0.01], # Learning rate
        'max_depth': [3, 5, 10, 15],       # Maximum depth of individual trees
        'subsample': [0.8, 1.0],           # Fraction of samples to use for each tree
        'min_samples_split': [2, 5, 10],   # Minimum number of samples required to split a node
    },
    'SVR': {
        'C': [0.1, 1, 10, 100],            # Regularization parameter to control model complexity
        'kernel': ['linear', 'rbf', 'poly'], # Types of kernels: linear, rbf, poly
        'gamma': ['scale', 'auto'],        # Kernel coefficient options: scale or auto
        'epsilon': [0.01, 0.1, 0.2],       # Epsilon: margin of tolerance for predictions
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100],  # Regularization strength
        'max_iter': [1000, 5000, 10000],   # Maximum number of iterations in training
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1, 10, 100],  # Regularization strength
        'l1_ratio': [0.1, 0.5, 0.7, 1.0],  # Mixing parameter for L1 and L2 penalties in ElasticNet
        'max_iter': [1000, 5000, 10000],   # Maximum number of iterations in training
    }
}


##### **c. Using RandomizedSearchCV**

**RandomizedSearchCV** is a tool in the `scikit-learn` library that helps find the optimal hyperparameters for a model by randomly sampling a set of hyperparameter combinations within the specified search space. It automatically integrates **cross-validation** to evaluate model performance.

##### **Reasons to Use RandomizedSearchCV**
1. **Integrated Cross-Validation**: Automatically splits the data to accurately evaluate the model's performance.
2. **Optimized Time**: Compared to GridSearchCV (which tests all parameter combinations), RandomizedSearchCV reduces the number of trials by randomly selecting combinations.
3. **Diversified Search**: Enables testing of many different combinations without being limited by the exhaustive search of the entire parameter space.
4. **Suitable for Large Datasets**: Reduces training time while maintaining efficient search capabilities.


In [17]:
# RandomizedSearchCV for hyperparameter tuning
def tune_hyperparameters(model, param_grid, X, y, metrics='neg_mean_squared_error', cv=5):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        scoring=metrics,
        cv=cv,
        n_iter=20,  # Number of random combinations to test
        n_jobs=-1,
        verbose=1
    )
    random_search.fit(X, y)
    return random_search.best_params_, random_search.best_score_

**Tuning and Evaluation Process**:
   - Iterate through each model in the models list.
   - Check if the model has a defined hyperparameter space in `param_grids`. 
   - Tune the model's hyperparameters using an optimization method (`RandomizedSearchCV`).

   - Calculate the evaluation metrics:
     - **MSE (Mean Squared Error)**: The mean of the squared errors.
     - **RMSE (Root Mean Squared Error)**: The square root of MSE
   - Measure the training time and the maximum memory usage.

In [18]:
# Import time and memory profiler
import time
from memory_profiler import memory_usage

# Function to calculate RMSE from MSE
def compute_rmse(mse):
    if mse < 0:
        mse = -mse
    return np.sqrt(mse)

# Function to measure training time and memory usage
def train_and_measure_memory(model, X_train, y_train):
    start_time = time.time()
    mem_usage = memory_usage((model.fit, (X_train, y_train)))
    end_time = time.time()
    training_time = end_time - start_time
    max_memory = max(mem_usage)
    return training_time, max_memory

# Perform hyperparameter tuning and evaluation for each model
result_list = []
for model in models:
    model_name = model.__class__.__name__
    if model_name in param_grids:
        print(f"\nTuning hyperparameters for {model_name}...")
         # Hyperparameter tuning
        best_params, best_score = tune_hyperparameters(
            model, param_grids[model_name], X_scaled, y, metrics='neg_mean_squared_error', cv=5
        )

        # Calculate RMSE and measure resource usage
        best_mse = -best_score
        best_rmse = compute_rmse(best_mse)
        training_time, max_memory = train_and_measure_memory(model.set_params(**best_params), X_scaled, y)

        # Store the results
        result_list.append({
            'Model': model_name,
            'Best Params': best_params,
            'Best Score (MSE)': best_mse,
            'Best Score (RMSE)': best_rmse,
            'Training Time (s)': training_time,
            'Max Memory (MB)': max_memory
        })

        print(f"Best parameters for {model_name}: {best_params}")
        print(f"Best score (MSE) for {model_name}: {best_mse}")
        print(f"Best score (RMSE) for {model_name}: {best_rmse}")
        print(f"Training time for {model_name}: {training_time:.4f} seconds")
        print(f"Max memory usage for {model_name}: {max_memory:.2f} MB")

# Convert results to DataFrame for display
results_df = pd.DataFrame(result_list)

# Display the hyperparameter tuning results
print("\nKết quả hyperparameter tuning:")
print(results_df.sort_values(by='Best Score (MSE)', ascending=True).reset_index(drop=True))


Tuning hyperparameters for RandomForestRegressor...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters for RandomForestRegressor: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 50, 'bootstrap': True}
Best score (MSE) for RandomForestRegressor: 0.13524912899652322
Best score (RMSE) for RandomForestRegressor: 0.3677623267771228
Training time for RandomForestRegressor: 1.6031 seconds
Max memory usage for RandomForestRegressor: 131.00 MB

Tuning hyperparameters for GradientBoostingRegressor...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters for GradientBoostingRegressor: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.01}
Best score (MSE) for GradientBoostingRegressor: 0.13989883032057054
Best score (RMSE) for GradientBoostingRegressor: 0.37403052057361647
Training time for GradientBoostingRegressor: 1.3402 seconds
Max memory usage for GradientBoos

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 15 is smaller than n_iter=20. Running 15 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters for Ridge: {'max_iter': 1000, 'alpha': 100}
Best score (MSE) for Ridge: 0.13060327665401034
Best score (RMSE) for Ridge: 0.36139075341520616
Training time for Ridge: 1.9744 seconds
Max memory usage for Ridge: 137.06 MB

Tuning hyperparameters for ElasticNet...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters for ElasticNet: {'max_iter': 5000, 'l1_ratio': 0.1, 'alpha': 0.01}
Best score (MSE) for ElasticNet: 0.1306519496037899
Best score (RMSE) for ElasticNet: 0.36145808830871373
Training time for ElasticNet: 1.9913 seconds
Max memory usage for ElasticNet: 137.28 MB

Kết quả hyperparameter tuning:
                       Model  \
0                      Ridge   
1                 ElasticNet   
2                        SVR   
3      RandomForestRegressor   
4  GradientBoostingRegressor   

                                         Best Params  Best Score (MSE)  \
0                   {'max_iter': 1000, 'alpha': 100}          0.130603   
1  {'max_it

### **Nhận xét**
1. **Performance (RMSE):**
   - The **Ridge** model achieves the lowest RMSE (0.3613), indicating the best performance in predictions.
   - **ElasticNet** and **SVR** have RMSE results similar to Ridge, with only a negligible difference (0.3614 and 0.3616).
   - Ensemble models like **RandomForestRegressor** and **GradientBoostingRegressor** have higher RMSE (0.3677 & 0.3740), suggesting their prediction capability is not superior to linear models.

2. **Training Time:**
   - **GradientBoostingRegressor** is the fastest model with a training time of ~0.34 seconds.
   - **Ridge**, **SVR** and **ElasticNet** take more time (1.974, 1.979 and 1.991 seconds), due to optimization processes and more complex structures.

3. **Memory Usage:**
   - The memory usage across models does not differ significantly. **RandomForestRegressor** has the lowest usage (~131.003 MB), while **ElasticNet** uses the most (~137.281 MB).

### **Conclusion**
- **Ridge** Ridge is the best model for the current dataset, due to its high prediction performance and balance between training time and memory usage.
- **ElasticNet** and **SVR** are also viable options if there is a need to reduce model complexity or training time.
- Ensemble models like **RandomForestRegressor** and **GradientBoostingRegressor** do not outperform linear models in this case, but they are still worth considering if there is a need to enhance the ability to learn non-linear relationships.

##### **Bảng kết quả chi tiết**

In [19]:
# Select relevant columns and sort by MSE
summary_df = results_df[['Model', 'Best Score (MSE)', 'Best Score (RMSE)', 'Training Time (s)', 'Max Memory (MB)']]
summary_df_sorted = summary_df.sort_values(by='Best Score (MSE)', ascending=True).reset_index(drop=True)

# Display the summary of results
print("\nKết quả tổng quan các mô hình:")
print(summary_df_sorted)

# Display results in a pretty table format
from tabulate import tabulate

print("\nBảng kết quả chi tiết:")
print(tabulate(summary_df_sorted, headers='keys', tablefmt='pretty', showindex=False))


Kết quả tổng quan các mô hình:
                       Model  Best Score (MSE)  Best Score (RMSE)  \
0                      Ridge          0.130603           0.361391   
1                 ElasticNet          0.130652           0.361458   
2                        SVR          0.130771           0.361622   
3      RandomForestRegressor          0.135249           0.367762   
4  GradientBoostingRegressor          0.139899           0.374031   

   Training Time (s)  Max Memory (MB)  
0           1.974427       137.062500  
1           1.991279       137.281250  
2           1.979634       136.171875  
3           1.603135       131.003906  
4           1.340231       132.652344  

Bảng kết quả chi tiết:
+---------------------------+---------------------+---------------------+--------------------+-----------------+
|           Model           |  Best Score (MSE)   |  Best Score (RMSE)  | Training Time (s)  | Max Memory (MB) |
+---------------------------+---------------------+------------

### **Choose the best model and predict**

- The best model is selected based on the lowest Mean RMSE and MSE values ​​from evaluation results.
- After choosing the best model, retrain the model on the entire data set to increase accuracy, then predict **Rating** for products without information and save the results Go to the new column **Predicted Rating**

In [21]:
# Select the best model based on MSE
best_model_info = results_df.loc[results_df['Best Score (MSE)'].idxmin()]
best_model_name = best_model_info['Model']
best_params = best_model_info['Best Params']

# Display the best model's information
print(f"\nBest Algorithm: {best_model_name}")
print(f"Best MSE Score: {best_model_info['Best Score (MSE)']}")
print(f"Best RMSE Score: {best_model_info['Best Score (RMSE)']}")
print(f"Training Time: {best_model_info['Training Time (s)']:.4f} seconds")
print(f"Max Memory Usage: {best_model_info['Max Memory (MB)']:.2f} MB")

# Initialize the best model with optimal parameters
best_model = eval(best_model_name)().set_params(**best_params)

# Train the best model on the full training data
best_model.fit(X_scaled, y)

# Make predictions on the unknown data (X_test)
predictions = best_model.predict(scaler.transform(X_test))

# Store and display the predictions
prediction_data['Predicted Rating'] = predictions
print("\nPrediction results on unknown dataset:")
print(prediction_data[['Price', 'Trademark', 'Country', 'General_function', 'Predicted Rating']])


Best Algorithm: Ridge
Best MSE Score: 0.13060327665401034
Best RMSE Score: 0.36139075341520616
Training Time: 1.9744 seconds
Max Memory Usage: 137.06 MB

Prediction results on unknown dataset:
         Price  Trademark  Country  General_function  Predicted Rating
3     390000.0        375       23                 0          4.947837
10     22400.0        214       38                 3          4.920278
14    151200.0        290       38                 0          4.921739
30     39000.0         14       38                 0          4.900257
31     39000.0         14       38                 0          4.900257
...        ...        ...      ...               ...               ...
1994  276000.0        208       38                14          4.861462
1995  276000.0        342       38                 4          4.885514
1996  276000.0        342       38                15          4.934686
1997  276000.0        234        7                 5          4.946959
1998  276000.0        157

C:\Users\Admin\AppData\Local\Temp\ipykernel_6680\3618910937.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prediction_data['Predicted Rating'] = predictions


### **Comments on the Results**

1. Best Model: Ridge

* The Ridge model has demonstrated the best performance in this scenario, with the following results:
    * Best MSE Score: 0.1306
    * Best RMSE Score: 0.36139
    * Training Time: 1.9744 seconds
    * Maximum Memory Usage: 131.06 MB
These results show that the Ridge model achieves a relatively low MSE and RMSE, indicating good prediction accuracy while maintaining a reasonable training time and memory usage.

2. Prediction Results on Unseen Data:
    According to the comparison of **Rating** on the newly updated Long Chau website, the prediction is quite stable and accurate.

### **Conclusion on the best model**

* The Ridge model stands out as the most effective model for this task due to its balanced performance in terms of prediction accuracy, training time, and memory usage.
* The predictions on the unseen data are within the expected range, indicating that the model is generalizing well.